In [1]:
!pip install selenium
!pip install unidecode
!pip install fastparquet
!pip install google-colab-selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 9.8 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 60.0 MB/s eta 0:00:00


In [2]:
#set mode

#mode = 'get_todays_races'
#mode = 'get_past_results'
mode = 'get_both'

In [3]:
# from google.colab import drive
# drive.mount('/content/drive')

In [4]:
from selenium import webdriver
from datetime import date, datetime, timedelta
from selenium.webdriver.common.by import By
import pytz
from bs4 import BeautifulSoup
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
import json
import re
import pandas as pd
from unidecode import unidecode
import numpy as np
import sys
import fastparquet
import google_colab_selenium as gs

In [5]:
def web_driver():
    driver = gs.Chrome()
    return driver

def get_web_content(url, driver=None):
  # If no driver is provided, create a new one
  if driver is None:
    local_driver = web_driver()
    should_quit_driver = True
  else:
    local_driver = driver
    should_quit_driver = False

  local_driver.get(url)

  # Locate the script tag with id="__NEXT_DATA__"
  script_tag = local_driver.find_element(By.ID, "__NEXT_DATA__")

  # Get the content of the script tag
  script_content = script_tag.get_attribute('innerHTML')

  # Parse the JSON content
  json_data = json.loads(script_content)

  # Access the "props" object
  props = json_data

  # Close the browser session if it was created locally
  if should_quit_driver:
    local_driver.quit()

  return props


def get_operator_data_odds(data, operators_priority=['PMU', 'PMU.fr', 'genybet']):
    for operator in operators_priority:
        for race in data:
            if race.get('operator') == operator:
              return race.get('runners', {})
    return None  # In case none of the operators are found

def get_operator_data_dividends(data, operators_priority=['PMU', 'PMU.fr', 'genybet']):
    for operator in operators_priority:
        for race in data:
            if race.get('operator') == operator:
              return race.get('betDividends', {})
    return None  # In case none of the operators are found

def generate_top_5_table(df, group_column):
    # Step 1: Filter for wins (ranking == 1)
    df_wins = df[df['ranking'] == 1]

    # Step 2: Count total runs for each group (jockey or trainer)
    total_runs = df.groupby(group_column).size()

    # Step 3: Count wins for each group
    win_count = df_wins.groupby(group_column).size()

    # Step 4: Merge wins and total runs, fill missing win counts with 0
    result = pd.DataFrame({
        'wins': win_count,
        'runs': total_runs
    }).fillna(0)

    # Step 5: Calculate win percentage
    result['win_percentage'] = (result['wins'] / result['runs']) * 100

    # Step 6: Sort by wins and select top 5
    top_5 = result.sort_values(by='wins', ascending=False).head(20)

    # Step 7: Format the result
    top_5['formatted'] = top_5.apply(
        lambda row: f"{row.name} {int(row['runs'])}/{int(row['wins'])} {row['win_percentage']:.0f}%", axis=1
    )

    return top_5['formatted']

In [6]:
# Define a function for the code you want to repeat
def repeatable_cells():
    # Add the code from the cells you want to repeat here

    # Initialize the driver once for this repeatable cell execution
    driver = web_driver()

    try:
        # File paths
        races_path = "/content/drive/MyDrive/PT/races.parquet"
        tracker_path = "/content/drive/MyDrive/PT/reload_tracker.csv"

        # Load existing races
        races = pd.read_parquet(races_path)

        # Determine the start date as the day after the last race date
        date_str = races.date.max()
        date_obj = datetime.strptime(date_str, '%Y-%m-%d')
        next_day = date_obj + timedelta(days=1)
        start_date = next_day.strftime('%Y-%m-%d')

        # Determine yesterday
        yesterday = datetime.today() - timedelta(days=1)
        yesterday_str = yesterday.strftime('%Y-%m-%d')


        # Load or initialize reload tracker
        try:
            reload_tracker_df = pd.read_csv(tracker_path, dtype={'reload_started': str, 'reload_finished': str})
        except FileNotFoundError:
            reload_tracker_df = pd.DataFrame(columns=['date', 'reload_done', 'reload_started', 'reload_finished', 'races_pushed'])

        # Add missing dates between start_date and yesterday
        def generate_date_list(start_date_str, end_date_str):
            start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
            end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
            date_list = []

            current_date = start_date
            while current_date <= end_date:
                date_list.append(current_date.strftime("%Y-%m-%d"))
                current_date += timedelta(days=1)

            return date_list

        # Generate list of missing dates
        missing_dates = generate_date_list(start_date, yesterday_str)

        # Add only dates that are not already in the tracker
        for date in missing_dates:
            if date not in reload_tracker_df['date'].values:
                new_row = {
                    'date': date,
                    'reload_done': False,
                    'reload_started': '9999-12-31',
                    'reload_finished': '9999-12-31',
                    'races_pushed': np.nan
                }
                reload_tracker_df = pd.concat([reload_tracker_df, pd.DataFrame([new_row])], ignore_index=True)


        # Save the updated tracker
        reload_tracker_df.to_csv(tracker_path, index=False)

        pending = reload_tracker_df[reload_tracker_df['reload_done'] == False]

        if pending.empty:
            print("reload complete")
            return False  # signal to stop the loop

        # # # Save DataFrame to CSV
        # rtpath = "/content/drive/MyDrive/PT/reload_tracker.csv"
        # reload_tracker_df.to_csv(rtpath, index=False)

        # # Reload the DataFrame from the CSV
        # reload_tracker_df = pd.read_csv(rtpath)

        # Get the latest date where reload_done is False
        loadDate = reload_tracker_df[reload_tracker_df['reload_done'] == False]['date'].min()
        url = 'https://www.paris-turf.com/programme-courses/' + loadDate

        # Get current timestamp
        my_tz = pytz.timezone('Europe/Berlin')
        timestamp_start = datetime.now(my_tz).strftime('%Y-%m-%d %H:%M:%S')

        # Update the reload_started column with the timestamp
        reload_tracker_df.loc[reload_tracker_df['date'] == loadDate, 'reload_started'] = timestamp_start

        print('loaddate:', loadDate, 'progress:', round((reload_tracker_df[reload_tracker_df['reload_done']==True].shape[0]/reload_tracker_df.shape[0])*100,2),'%')
        print("Running the repeatable code")
        # e.g., other computations, plots, etc.

        # Extract arrays

        props = get_web_content(url, driver)

        races = props['props']['pageProps']['initialState']['raceCardsState']['races'][loadDate]
        meetings = props['props']['pageProps']['initialState']['raceCardsState']['meetings'][loadDate]

        # Convert to DataFrames
        df_races = pd.DataFrame(races)
        df_meetings = pd.DataFrame(meetings)

        # Merge DataFrames on meetingId
        merged_df = pd.merge(df_meetings[['id', 'name', 'country']], df_races, left_on='id', right_on='meetingId', suffixes=('_meeting', '_race'))


        # Filter based on conditions
        filtered_df = merged_df[(merged_df['country'] == 'FR') & (merged_df['discipline'] != 'Trot') & (merged_df['specialty'] == 'P') & (merged_df['breed'].isnull() == True)]

        # Define the required columns
        required_columns = ['date', 'name_meeting', 'meetingId', 'trackCode', 'name_race', 'id_race', 'specialty', 'class', 'type', 'sex',
                            'surface', 'going', 'penetrometer', 'number', 'distance', 'isPremium', 'breed', 'totalPrize',
                            'uuid', 'direction', 'rail', 'winningPost', 'minAge', 'maxAge', 'discipline', 'winnerTimeKm', 'winnerTime']

        # Reindex the DataFrame to ensure all required columns are present, filling missing ones with NaN
        final_df = filtered_df.reindex(columns=required_columns, fill_value=np.nan)

        def clean_name(name):
            # Remove accents and convert to lowercase
            name = unidecode(name.lower())
            # Replace spaces and parentheses with hyphens
            name = name.replace(' ', '-').replace('(', '').replace(')', '')
            return name

        # Apply the cleaning function to 'name_race' and 'name_meeting'
        final_df['cleaned_name_race'] = final_df['name_race'].apply(clean_name)
        final_df['cleaned_name_meeting'] = final_df['name_meeting'].apply(clean_name)

        # Construct the 'race_url' column
        final_df['race_url'] = 'https://www.paris-turf.com/course/' + final_df['cleaned_name_meeting'] + '-' + final_df['cleaned_name_race'] + '-idc-' + final_df['uuid']

        # Drop intermediate columns if you don't need them
        final_df.drop(columns=['cleaned_name_race', 'cleaned_name_meeting'], inplace=True)
        race_df = final_df.reset_index(drop=True)
        race_df['runners_loaded'] = False

        path = "/content/drive/MyDrive/PT/races.parquet"

        # # Read the existing Parquet file
        df_races_old = pd.read_parquet(path)

        # Append the new data
        df_combined = pd.concat([df_races_old, race_df])

        # Write the updated DataFrame back to Parquet
        df_combined.to_parquet(path,  engine='pyarrow', index=None)
        df_races_old = pd.read_parquet(path)

        runners_dfs = []
        webTips_dfs = []
        odds_dfs = []
        dividends_dfs = []
        skipped_races = []


        for i in range(len(race_df['race_url'])):
            props = get_web_content(race_df['race_url'][i], driver)

            # Extract runners (this part is assumed to always exist)
            runners = props['props']['pageProps']['initialState']['raceCardsState']['runners'][str(race_df['id_race'][i])]
            df_runners = pd.DataFrame(runners)

            # Safely attempt to extract webTips using get() at each level
            try:
                webTips = props['props']['pageProps']['initialState'].get('currentPageState', {}).get('webTips', [])
                df_webTips = pd.DataFrame(webTips).reset_index()
            except:
                webTips = pd.DataFrame()


            # Safely attempt to extract webTips using get() at each level
            try:
                odds = get_operator_data_odds(props['props']['pageProps']['initialState'].get('currentPageState', {}).get('betinRaceOdd', {}).get('odds', {}))
                df_odds = pd.DataFrame(odds).reset_index()
            except:
                df_odds = pd.DataFrame()

            # Safely attempt to extract webTips using get() at each level
            try:
                dividends = get_operator_data_dividends(props['props']['pageProps']['initialState'].get('currentPageState', {}).get('dividends', {}).get('raceDividends', {}))
                df_dividends = pd.DataFrame(dividends).reset_index()
            except:
                df_dividends = pd.DataFrame()

            # Define the keys for runners and webTips
            keys = ['horseId', 'isRunnerState', 'meetingId', 'age', 'hood', 'breederName', 'shoeingFront', 'isEngaged', 'jockeyName', 'draw', 'saddle', 'isSupplemented', 'isRunning', 'weightKg', 'raceDirection', 'numberOfPlaces', 'ownerName', 'raceId', 'uuid', 'raceSpeciality', 'noShoesFirstTime',
                    'raceTotalPrize', 'ranking', 'jockeyUUID', 'totalPrize', 'horseName', 'horseSir', 'trainerUUID', 'meetingName', 'horseUUID', 'ownerUUID', 'trainerName', 'protectionFirstTime', 'raceName', 'jockeyAllowance', 'tongueTie', 'raceIsTQQ', 'sex', 'shoeingBack',
                    'totalWinningPrize','breederId', 'margin', 'jockeyChanged', 'coloursPng', 'shoeing', 'bestImpression', 'comment', 'isPremium', 'blinkers', 'jockeyId', 'raceType', 'ownerId', 'handicapRatingKg', 'horseDam', 'weightChanged', 'claimRating',
                    'trainerId', 'blinkersFirstTime','raceNumber']

            keys2 = ['meetingId', 'raceId', 'text', 'tips']

            keys_odds = ['horseId', 'horseNumber', 'liveOdd', 'referenceOdd', 'isFavorite', 'runnerId', 'liveOddDateTime', 'referenceOddDateTime', 'runnerStatus', 'runnerSlug', 'horseName']

            keys_dividends = ['betType',    'turnover'  ,'dividend' ,'originDividendType'   ,'minimumBet'   ,'dividendType' ,'originBetType',   'combination']

            # Reindex runners and webTips dataframes
            if not df_runners.empty:
                df_runners_required_columns = df_runners.reindex(columns=keys, fill_value=np.nan)

            if not df_webTips.empty:
                df_webTips_required_columns = df_webTips.reindex(columns=keys2, fill_value=np.nan)

            if not df_odds.empty:
                df_odds_required_columns = df_odds.reindex(columns=keys_odds, fill_value=np.nan)
                df_odds_required_columns['meetingId'] = race_df['meetingId'][i]
                df_odds_required_columns['raceId'] = race_df['id_race'][i]

            if not df_dividends.empty:
                df_dividends_required_columns = df_dividends.reindex(columns=keys_dividends, fill_value=np.nan)
                #desired_types = ['Gagnant', 'Placé', 'Ordre', 'Multi en 4', 'Multi en 5', 'Multi en 6', 'Multi en 7']
                df_dividends_required_columns['meetingId'] = race_df['meetingId'][i]
                df_dividends_required_columns['raceId'] = race_df['id_race'][i]
                #df_dividends_required_columns_filtered = df_dividends_required_columns[df_dividends_required_columns['originDividendType'].isin(desired_types)]

            # Append the runners to the list
            runners_dfs.append(df_runners_required_columns)

            # Append webTips only if it exists
            if not df_webTips.empty:
                webTips_dfs.append(df_webTips_required_columns)

            # Append webTips only if it exists
            if not df_odds.empty:
                odds_dfs.append(df_odds_required_columns)

            # Append webTips only if it exists
            if not df_dividends.empty:
                dividends_dfs.append(df_dividends_required_columns)

            # Print the race URL and shape of the runners dataframe
            print(race_df['race_url'][i], df_runners_required_columns.shape)

            # Check if the number of runners is less than 2
            if df_runners_required_columns.shape[0] < 2:
                print(f'⚠️  Skipping — only {df_runners_required_columns.shape[0]} runner(s), url={race_df["race_url"][i]}')
                skipped_races.append({'url': race_df['race_url'][i], 'runners': df_runners_required_columns.shape[0]})
                continue


        path = "/content/drive/MyDrive/PT/runners.parquet"

        if len(runners_dfs) != 0:

            df_runners_new = pd.concat(runners_dfs, ignore_index=True)
            # Read the existing Parquet file
            df_runners_old = pd.read_parquet(path)
            # Append the new data
            df_combined = pd.concat([df_runners_new, df_runners_old], ignore_index=True)

            # Write the updated DataFrame back to Parquet
            df_combined.to_parquet(path,  engine='fastparquet', index=None)
            print('old:', df_runners_old.shape[0],'+ new:', df_runners_new.shape[0],'= ', df_combined.shape[0])

            # Generate tables for jockeyName and trainerName
            top_5_jockeys = generate_top_5_table(df_combined, 'jockeyName')
            top_5_trainers = generate_top_5_table(df_combined, 'trainerName')
            top_5_owners = generate_top_5_table(df_combined, 'ownerName')
            top_5_breeders = generate_top_5_table(df_combined, 'breederName')
            top_5_horses = generate_top_5_table(df_combined, 'horseName')
            top_5_sires = generate_top_5_table(df_combined, 'horseSir')

            # Print the final result
            print("Top 20 Jockeys:")
            print(top_5_jockeys.to_string(index=False))

            print("\nTop 20 Trainers:")
            print(top_5_trainers.to_string(index=False))

            print("\nTop 20 Owners:")
            print(top_5_owners.to_string(index=False))

            print("\nTop 20 Breeders:")
            print(top_5_breeders.to_string(index=False))

            print("\nTop 20 Horses:")
            print(top_5_horses.to_string(index=False))

            print("\nTop 20 Sires:")
            print(top_5_sires.to_string(index=False))


            path = "/content/drive/MyDrive/PT/webTips.parquet"

        if len(webTips_dfs) != 0:

            df_webTips_new = pd.concat(webTips_dfs, ignore_index=True)

            # Read the existing Parquet file
            df_webTips_old = pd.read_parquet(path)
            # Append the new data
            df_combined = pd.concat([df_webTips_new, df_webTips_old], ignore_index=True)
            # Write the updated DataFrame back to Parquet
            df_combined.to_parquet(path, engine='pyarrow', index=None)
            #print('old:', df_webTips_old.shape[0],'+ new:', df_webTips_new.shape[0],'= ', df_combined.shape[0])


            path = "/content/drive/MyDrive/PT/odds.parquet"

        if len(odds_dfs) != 0:

            df_odds_new = pd.concat(odds_dfs, ignore_index=True)

            # Read the existing Parquet file
            df_odds_old = pd.read_parquet(path)

            # Append the new data
            df_combined = pd.concat([df_odds_new, df_odds_old], ignore_index=True)

            # Write the updated DataFrame back to Parquet
            df_combined.to_parquet(path,  engine='pyarrow', index=None)
            #print('old:', df_odds_old.shape[0],'+ new:', df_odds_new.shape[0],'= ', df_combined.shape[0])


            path = "/content/drive/MyDrive/PT/dividends.parquet"

        if len(dividends_dfs) != 0:

            df_dividends_new = pd.concat(dividends_dfs, ignore_index=True)

            # Read the existing Parquet file
            df_dividends_old = pd.read_parquet(path)

            # Append the new data
            df_combined = pd.concat([df_dividends_new, df_dividends_old], ignore_index=True)

            # Write the updated DataFrame back to Parquet
            df_combined.to_parquet(path,  engine='pyarrow', index=None)
            #print('old:', df_dividends_old.shape[0],'+ new:', df_dividends_new.shape[0],'= ', df_combined.shape[0])

        timestamp_finished = datetime.now(my_tz).strftime('%Y-%m-%d %H:%M:%S')
        races_cnt =  race_df['id_race'].nunique()
        reload_tracker_df.loc[reload_tracker_df['date'] == loadDate, 'reload_done'] = True
        reload_tracker_df.loc[reload_tracker_df['date'] == loadDate, 'reload_finished'] = timestamp_finished
        reload_tracker_df.loc[reload_tracker_df['date'] == loadDate, 'races_pushed'] = races_cnt
        reload_tracker_df.to_csv(tracker_path, index=False)

        return True
    finally:
        driver.quit() # Ensure the driver is quit even if errors occur

In [ ]:
# Run the function multiple times
if mode in ('get_past_results', 'get_both'):
  for i in range(10):
      print(f"Iteration {i+1}")
      if not repeatable_cells():
          break

Iteration 1


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

loaddate: 2026-04-21 progress: 99.69 %
Running the repeatable code


...

https://www.paris-turf.com/course/le-mans-prix-du-canton-d'ecommoy-idc-12cddf8543e12674fac272982024a48f (10, 59)
https://www.paris-turf.com/course/le-mans-prix-de-la-societe-des-courses-de-chateau-du-loir-idc-c01e6cb14d7a1bb4cdd7c6321082db9c (15, 59)
https://www.paris-turf.com/course/le-mans-prix-du-canton-de-la-suze-sur-sarthe-idc-48a0ff9f7ab0a681fb6e277048884b90 (14, 59)
https://www.paris-turf.com/course/le-mans-prix-du-canton-de-la-fleche-idc-6c991211c38efa2b3bff05348adbb321 (9, 59)
https://www.paris-turf.com/course/le-mans-prix-du-canton-de-sable-sur-sarthe-idc-a1eb1694b3be24b90a5774f8edce9484 (15, 59)
https://www.paris-turf.com/course/le-mans-prix-de-la-societe-des-courses-de-montmirail-idc-59b23a48876e379dddefab0979049ac1 (15, 59)
https://www.paris-turf.com/course/le-mans-prix-de-pruille-le-chetif-idc-22268c80c48b9c42f3c449fd067a4752 (15, 59)
https://www.paris-turf.com/course/le-mans-prix-du-canton-de-sille-le-guillaume-idc-797d76c5cc5aec51aab27fd2f167843e (15, 59)


/tmp/ipykernel_1255/3835817399.py:243: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_runners_new = pd.concat(runners_dfs, ignore_index=True)


old: 315674 + new: 108 =  315782
Top 20 Jockeys:
jockeyName
      M. Guyon 8500/1307 15%
  M. Barzalona 6377/1114 17%
      C. Demuro 6942/936 13%
   C. Soumillon 4393/708 16%
    T. Bachelot 6479/706 11%
     A. Pouchin 5087/545 11%
     A. Lemaitre 6051/536 9%
    S. Pasquier 4786/531 11%
   Mlle M. Vélon 6008/524 9%
       A. Orani 3476/524 15%
      A. Crastus 6402/502 8%
     T. Piccone 5069/491 10%
     E. Hardouin 5875/426 7%
      A. Madamet 4470/397 9%
     M. Grandin 3300/396 12%
    C. Lecoeuvre 4205/378 9%
   I. Mendizabal 4720/370 8%
     PC. Boudot 1614/358 22%
Mlle D. Santiago 3547/292 8%
     JB. Eyquem 1211/286 24%

Top 20 Trainers:
trainerName
JC. Rouget (s) 3615/863 24%
      A. Fabre 3740/721 19%
  FH. Graffard 3286/711 22%
   HA. Pantall 5410/656 12%
    C. Escuder 6362/618 10%
    J. Reynier 3405/598 18%
    C. Ferland 2454/409 17%
   Y. Barberot 3015/359 12%
  F. Vermeulen 3239/355 11%
    P. Cottier 2256/327 14%
     S. Wattel 2431/322 13%
    F. Chappet 2532/31

<IPython.core.display.Javascript object>

reload complete


In [ ]:
def get_today():

    # Initialize the driver once for this get_today execution
    driver = web_driver()

    try:
        loadDate = datetime.today().strftime('%Y-%m-%d')
        url = f'https://www.paris-turf.com/programme-courses/{loadDate}'
        my_tz = pytz.timezone('Europe/Berlin')
        timestamp_start = datetime.now(my_tz).strftime('%Y-%m-%d %H:%M:%S')

        print('loadDate:', loadDate)
        print("Running today's data collection...")

        props = get_web_content(url, driver)
        races = props['props']['pageProps']['initialState']['raceCardsState']['races'][loadDate]
        meetings = props['props']['pageProps']['initialState']['raceCardsState']['meetings'][loadDate]

        df_races_tdy = pd.DataFrame(races)
        df_meetings_tdy = pd.DataFrame(meetings)

        merged_df = pd.merge(
            df_meetings_tdy[['id', 'name', 'country']],
            df_races_tdy,
            left_on='id',
            right_on='meetingId',
            suffixes=('_meeting', '_race')
        )

        filtered_df = merged_df[
            (merged_df['country'] == 'FR') &
            (merged_df['discipline'] != 'Trot') &
            (merged_df['specialty'] == 'P') &
            (merged_df['breed'].isnull())
        ]

        required_columns = ['date', 'time', 'name_meeting', 'meetingId', 'trackCode', 'name_race', 'id_race', 'specialty', 'class', 'type', 'sex',
                            'surface', 'going', 'penetrometer', 'number', 'distance', 'isPremium', 'breed', 'totalPrize',
                            'uuid', 'direction', 'rail', 'winningPost', 'minAge', 'maxAge', 'discipline', 'winnerTimeKm', 'winnerTime']

        final_df = filtered_df.reindex(columns=required_columns, fill_value=np.nan)

        def clean_name(name):
            name = unidecode(name.lower())
            name = name.replace(' ', '-').replace('(', '').replace(')', '')
            return name

        final_df['cleaned_name_race'] = final_df['name_race'].apply(clean_name)
        final_df['cleaned_name_meeting'] = final_df['name_meeting'].apply(clean_name)
        final_df['race_url'] = 'https://www.paris-turf.com/course/' + final_df['cleaned_name_meeting'] + '-' + final_df['cleaned_name_race'] + '-idc-' + final_df['uuid']
        final_df.drop(columns=['cleaned_name_race', 'cleaned_name_meeting'], inplace=True)

        race_df_tdy = final_df.reset_index(drop=True)
        race_df_tdy['runners_loaded'] = False
        race_df_tdy.to_parquet("/content/drive/MyDrive/PT/races_tdy.parquet", engine='pyarrow', index=None)

        runners_dfs = []
        webTips_dfs = []
        odds_dfs = []
        skipped_races = []


        for i in range(len(race_df_tdy)):
            props = get_web_content(race_df_tdy['race_url'][i], driver)


            # Runners
            runners = props['props']['pageProps']['initialState']['raceCardsState']['runners'][str(race_df_tdy['id_race'][i])]
            df_runners = pd.DataFrame(runners)

            # WebTips (optional)
            try:
                webTips = props['props']['pageProps']['initialState'].get('currentPageState', {}).get('webTips', [])
                df_webTips = pd.DataFrame(webTips).reset_index()
            except:
                df_webTips = pd.DataFrame()

            # Odds (optional)
            try:
                odds = get_operator_data_odds(
                    props['props']['pageProps']['initialState'].get('currentPageState', {}).get('betinRaceOdd', {}).get('odds', {})
                )
                df_odds = pd.DataFrame(odds).reset_index()
            except:
                df_odds = pd.DataFrame()

            # Define the keys for runners and webTips
            keys = ['horseId', 'isRunnerState', 'meetingId', 'age', 'hood', 'breederName', 'shoeingFront', 'isEngaged', 'jockeyName', 'draw', 'saddle', 'isSupplemented', 'isRunning', 'weightKg', 'raceDirection', 'numberOfPlaces', 'ownerName', 'raceId', 'uuid', 'raceSpeciality', 'noShoesFirstTime',
                    'raceTotalPrize', 'ranking', 'jockeyUUID', 'totalPrize', 'horseName', 'horseSir', 'trainerUUID', 'meetingName', 'horseUUID', 'ownerUUID', 'trainerName', 'protectionFirstTime', 'raceName', 'jockeyAllowance', 'tongueTie', 'raceIsTQQ', 'sex', 'shoeingBack',
                    'totalWinningPrize','breederId', 'margin', 'jockeyChanged', 'coloursPng', 'shoeing', 'bestImpression', 'comment', 'isPremium', 'blinkers', 'jockeyId', 'raceType', 'ownerId', 'handicapRatingKg', 'horseDam', 'weightChanged', 'claimRating',
                    'trainerId', 'blinkersFirstTime','raceNumber']

            keys2 = ['meetingId', 'raceId', 'text', 'tips']

            keys_odds = ['horseId', 'horseNumber', 'liveOdd', 'referenceOdd', 'isFavorite', 'runnerId', 'liveOddDateTime', 'referenceOddDateTime', 'runnerStatus', 'runnerSlug', 'horseName']

            if not df_runners.empty:
                df_runners = df_runners.reindex(columns=keys, fill_value=np.nan)
                runners_dfs.append(df_runners)

            if not df_webTips.empty:
                df_webTips = df_webTips.reindex(columns=keys2, fill_value=np.nan)
                webTips_dfs.append(df_webTips)

            if not df_odds.empty:
                df_odds = df_odds.reindex(columns=keys_odds, fill_value=np.nan)
                df_odds['meetingId'] = race_df_tdy['meetingId'][i]
                df_odds['raceId'] = race_df_tdy['id_race'][i]
                odds_dfs.append(df_odds)

            print(race_df_tdy['race_url'][i], df_runners.shape)

            if df_runners.shape[0] < 2:
                print(f'⚠️  Skipping — only {df_runners.shape[0]} runner(s), url={race_df_tdy["race_url"][i]}')
                skipped_races.append({'url': race_df_tdy['race_url'][i], 'runners': df_runners.shape[0]})
                continue

        # Save runners
        if runners_dfs:
            df_runners_tdy = pd.concat(runners_dfs, ignore_index=True)
            df_runners_tdy.to_parquet("/content/drive/MyDrive/PT/runners_tdy.parquet", engine='fastparquet', index=None)

        # Save webTips
        if webTips_dfs:
            df_webTips_tdy = pd.concat(webTips_dfs, ignore_index=True)
            df_webTips_tdy.to_parquet("/content/drive/MyDrive/PT/webTips_tdy.parquet", engine='pyarrow', index=None)

        # Save odds
        if odds_dfs:
            df_odds_tdy = pd.concat(odds_dfs, ignore_index=True)
            df_odds_tdy.to_parquet("/content/drive/MyDrive/PT/odds_tdy.parquet", engine='pyarrow', index=None)

        return True
    finally:
        driver.quit() # Ensure the driver is quit even if errors occur

In [ ]:
get_today()